# AutoDL：低学习率优化 CMCR（Gated-ReZero seed1337）

冻结已验证的 A′ seed1337，只训练零初始化 CMCR；学习率5e-5，按 Val mIoU-FG 选模，不访问 Test。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_DIR = Path('/root/autodl-tmp/projects/lunar-linear/LTL-Net')
DATA_ROOT = Path('/root/autodl-tmp/datasets/dataset_v6_random811_overlap40')
OUTPUT_ROOT = Path('/root/autodl-tmp/outputs')
CONFIG_PATH = PROJECT_DIR / 'configs/v6_overlap40_frozen_gated_rezero_cmcr_lr5e5_batch4_seed1337.json'
# 如结果不在默认位置，只修改下一行。
BASE_CHECKPOINT = Path('/root/autodl-tmp/outputs/result_v6_overlap40_gated_rezero_resnet50_seed1337_bw0_batch4_valfg/best_model.pth')

config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['module'] == 'gated_rezero_cmcr' and config['seed'] == 1337
assert config['learning_rate'] == 5e-5 and config['epochs'] == 60
for path in (PROJECT_DIR, DATA_ROOT, BASE_CHECKPOINT):
    assert path.exists(), path
commit = subprocess.check_output(['git', '-C', str(PROJECT_DIR.parent), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Git commit:', commit)
print('父模型:', BASE_CHECKPOINT)
print('输出目录:', OUTPUT_ROOT / f"result_{config['run_name']}")
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
command = [sys.executable, str(PROJECT_DIR / 'scripts/run_autodl_frozen_rezero_cmcr.py'),
           '--project-dir', str(PROJECT_DIR), '--config', str(CONFIG_PATH),
           '--data-dir', str(DATA_ROOT), '--output-dir', str(OUTPUT_ROOT),
           '--init-checkpoint', str(BASE_CHECKPOINT)]
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
print(' '.join(command), flush=True)
subprocess.check_call(command, cwd=PROJECT_DIR, env=env)

In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
metrics = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('下载:', Path(str(result_dir) + '.zip'))